
# Transformer
---

In [1]:
# import requests
# from IPython.display import Image, display

# url = "https://drive.usercontent.google.com/download?id=1F1Fg--SX3kvCa0QIAysj5xOCesDuBP7a&export=download&authuser=0"
# response = requests.get(url)

# # Display using bytes content
# display(Image(data=response.content, height=700))

![](../images/E-D.svg)

![TR](../images/T-R.svg)

## Complete Transformer Shape Reference Guide

All shapes verified end-to-end.

---

### 1. Core Hyperparameters (Symbols Used Everywhere)

| Symbol | Meaning | "Base" Transformer (Vaswani et al.) |
|--------|---------|-------------------------------------|
| `B` | batch size | task-dependent |
| `L` (or `L_enc`, `L_dec`) | sequence length | task-dependent, e.g. 512 |
| `V` | vocabulary size | ~30k–50k (BPE) |
| `d_model` | model/embedding dimension | 512 |
| `h` | number of attention heads | 8 |
| `d_k` | dim per head for Q/K = `d_model / h` | 64 |
| `d_v` | dim per head for V (usually = `d_k`) | 64 |
| `d_ff` | FFN inner dimension | 2048 (= 4×`d_model`) |
| `N` | number of encoder/decoder layers | 6 |

> **Constraint:** `h × d_k = d_model` must always hold (so heads can be concatenated back to `d_model`).

---

### 2. Matrix Dimensions at Every Stage

#### Embedding + Positional Encoding

| Tensor | Shape |
|--------|-------|
| Token embedding table | `(V, d_model)` |
| Input token ids | `(B, L)` |
| Embedded input | `(B, L, d_model)` |
| Positional encoding | `(L, d_model)` — broadcasts over `B` |
| `X = embed + PE` | `(B, L, d_model)` |

#### Multi-Head Attention (Self or Cross)

| Tensor | Shape | Note |
|--------|-------|------|
| `Wq, Wk, Wv` | `(d_model, d_model)` | really `(d_model, h·d_k)`, split later |
| `Q = X·Wq` | `(B, L, d_model)` | for cross-attn, uses decoder `X` |
| `K = X·Wk`, `V = X·Wv` | `(B, L, d_model)` | for cross-attn, uses **encoder** output, so `L = L_enc` here |
| after head-split | `(B, h, L, d_k)` | reshape `(B,L,h,d_k)` → transpose |
| `scores = QKᵀ/√d_k` | `(B, h, L_q, L_k)` | **self-attn**: `L_q = L_k`. **cross-attn**: `L_q = L_dec`, `L_k = L_enc` — the one place the matrix is deliberately non-square |
| `attn = softmax(scores)` | `(B, h, L_q, L_k)` | rows sum to 1 |
| `head_out = attn·V` | `(B, h, L_q, d_v)` | |
| concat heads | `(B, L_q, h·d_v)` = `(B, L_q, d_model)` | transpose back, merge head dim |
| `Wo` | `(h·d_v, d_model)` | output projection |
| MHA output | `(B, L_q, d_model)` | same shape as the query input — enables stacking layers |

#### Feed-Forward Sublayer

| Tensor | Shape |
|--------|-------|
| `W1` | `(d_model, d_ff)` |
| `b1` | `(d_ff,)` |
| hidden = ReLU(`x·W1+b1`) | `(B, L, d_ff)` |
| `W2` | `(d_ff, d_model)` |
| `b2` | `(d_model,)` |
| FFN output | `(B, L, d_model)` |

#### LayerNorm

`γ, β` are both `(d_model,)`, applied per-token over the last axis. Input/output shape unchanged: `(B, L, d_model)`.

##### Output Head

| Tensor | Shape |
|--------|-------|
| `W_out` (often tied to embedding table transposed) | `(d_model, V)` |
| logits | `(B, L_dec, V)` |
| probs = softmax(logits) | `(B, L_dec, V)` |

---

##### The Key Invariant

> **Every sublayer takes `(B, L, d_model)` in and returns `(B, L, d_model)` out.**

This is what lets you stack `N` identical encoder/decoder layers arbitrarily deep — nothing about the shape changes layer to layer, only the content.

---

### 3. Common Calculations / Formulas

- **Attention complexity:** `O(L² · d_model)` — the `scores` matrix is `L×L` per head, the dominant cost. This is *the* reason long-context transformers are expensive and why alternatives (linear attention, sparse attention, sliding window) target this term.

- **Scaling factor:** `1/√d_k` — without it, dot products grow with `d_k` and push softmax into saturated (near one-hot / vanishing-gradient) regions. This connects to why `d_k` and `d_model` can't be picked independently of head count.

- **Parameter count per encoder layer** (roughly):
  - MHA: `Wq + Wk + Wv + Wo` → `4·d_model²`
  - FFN: `W1 + W2` → `2·d_model·d_ff` (= `8·d_model²` when `d_ff = 4·d_model`)
  - LayerNorm: `~4·d_model` (negligible)
  - Total ≈ `12·d_model²` per layer
  - Base model: `12 × 512² ≈ 3.1M` params/layer × 6 layers ≈ **~19M** for the encoder stack alone (decoder roughly doubles this since it has both self-attn and cross-attn)

- **Total model params** ≈ embedding (`V·d_model`, often tied with output projection) + `N·(encoder layer params)` + `N·(decoder layer params, ~1.5× encoder layer)`

- **Causal mask cost:** free in terms of extra params — it's just `L×L` boolean, applied additively as `-inf` before softmax.

---

### 4. Dry Run — Exact Shapes Traced Through One Full Pass

**Toy config:** `B=1, L_enc = L_dec = 4, d_model = 8, h = 2 → d_k = d_v = 4, d_ff = 16, V = 12`

```
Embedding + PE:
  token_ids_enc                   (1, 4)
  X_enc (embeds + PE)             (1, 4, 8)

Encoder self-attention:
  Wq/Wk/Wv                        (8, 8)
  Q = X @ Wq                      (1, 4, 8)
  Q_heads (after split)           (1, 2, 4, 4)     <- (B, h, L, d_k)
  scores = QK^T/sqrt(d_k)         (1, 2, 4, 4)     <- square, L_enc x L_enc
  head_out = attn @ V             (1, 2, 4, 4)
  concat heads                    (1, 4, 8)
  MHA output = concat @ Wo        (1, 4, 8)

Add & Norm + FFN:
  resid1 = X + MHA_out            (1, 4, 8)
  W1 (d_model -> d_ff)            (8, 16)
  ffn_hidden                      (1, 4, 16)
  W2 (d_ff -> d_model)            (16, 8)
  encoder_layer_out (final)       (1, 4, 8)

Decoder masked self-attention:
  X_dec (embeds + PE)             (1, 4, 8)
  dec_self_out                    (1, 4, 8)
  resid_d1                        (1, 4, 8)

Decoder cross-attention:
  Qc (from decoder)               (1, 2, 4, 4)
  Kc (from encoder, len=L_enc)    (1, 2, 4, 4)
  cross scores (L_dec x L_enc)    (1, 2, 4, 4)     <- would be non-square if L_enc != L_dec
  cross_out                       (1, 4, 8)
  decoder_layer_out (final)       (1, 4, 8)

Final projection:
  W_out (d_model -> vocab)        (8, 12)
  logits                          (1, 4, 12)
  probs                           (1, 4, 12)
```


## Attention
Attention is a mechanism that allows a model to decide which words are most important when understanding or generating text rather than treating all parts equally.
1. **Score each input.** For a given context (a Query), the model computes a raw relevance/compatibility score between that Query and every input element (Key). This score reflects how relevant each input is to what the model is currently focused on.

2. **Normalize with Softmax.** These raw scores are passed through a softmax function, converting them into a probability distribution — a set of weights $\alpha_i$ such that:
$$\sum_{i=1}^{n} \alpha_i = 1, \quad \alpha_i \ge 0$$

3. **Compute the context vector.** The final output — the context vector $\mathbf{c}$ — is a weighted sum of the input representations (Values), using the normalized weights:
$$\mathbf{c} = \sum_{i=1}^{n} \alpha_i \mathbf{v}_i$$

---
##### Summary

Given queries \(Q\), keys \(K\), and values \(V\), attention is commonly written as:

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Here:

| Symbol | Meaning |
|---|---|
| \(Q\) | Queries: what the model is looking for |
| \(K\) | Keys: labels or representations used for matching |
| \(V\) | Values: information retrieved after matching |
| \(d_k\) | Key-vector dimension |
| \(n\) | Number of query tokens |
| \(m\) | Number of key/value tokens |

For self-attention, usually \(n=m=L\), where \(L\) is the sequence length. The matrix $QK^T$ has size $n\times m$, so the main time complexity is:

> $O(nmd_k)+O(nmd_v)$

For self-attention with sequence length \(L\) and model dimension \(d\), this is generally expressed as:
$$O(L^2d)$$

> The quadratic $L^2$ term is the main scalability challenge of standard attention.

### Attention dimension

| Dimension                          | Main types                                          | What it answers                                           |
| ---------------------------------- | --------------------------------------------------- | --------------------------------------------------------- |
| **1. Score / similarity function** | Additive, Dot-Product, Scaled Dot-Product, Bilinear | *How do we calculate attention scores?*                   |
| **2. Attention relationship**      | Self, Cross, Encoder-Decoder                        | *What is attending to what?*                              |
| **3. Information-access pattern**  | Full, Causal, Bidirectional, Local, Sparse, Global  | *Which tokens are allowed to attend to which tokens?*     |
| **4. Computational architecture**  | MHA, MQA, GQA, MLA, Linear Attention                | *How do we make attention cheaper/more memory efficient?* |


### 1. Attention Score:
An attention score is a numerical value that measures how relevant one token is to another token.

<svg font-family="-apple-system-body, ui-sans-serif, -apple-system, system-ui, &quot;Segoe UI&quot;, Helvetica, &quot;Apple Color Emoji&quot;, Arial, sans-serif, &quot;Segoe UI Emoji&quot;, &quot;Segoe UI Symbol&quot;" font-weight="400" data-d-component="svg" fill="currentColor" style="color:rgb(255, 255, 255)" viewBox="0 0 720 120" xmlns="http://www.w3.org/2000/svg"><rect width="720" height="120" rx="14" fill="#F8FAFC"/><rect x="20" y="30" width="150" height="60" rx="10" fill="#DBEAFE" stroke="#1D4ED8"/><text x="95" y="48" font-size="11" font-family="Arial" font-weight="700" text-anchor="middle" fill="#1D4ED8">Query</text><text x="95" y="64" font-size="9" font-family="Arial" text-anchor="middle" fill="#1D4ED8">What I need</text><path d="M170 60 H195" stroke="#64748B" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="195" y="30" width="150" height="60" rx="10" fill="#FDE68A" stroke="#92400E"/><text x="270" y="48" font-size="11" font-family="Arial" font-weight="700" text-anchor="middle" fill="#92400E">Score</text><text x="270" y="64" font-size="9" font-family="Arial" text-anchor="middle" fill="#92400E">Similarity</text><path d="M345 60 H370" stroke="#64748B" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="370" y="30" width="150" height="60" rx="10" fill="#D1FAE5" stroke="#047857"/><text x="445" y="48" font-size="11" font-family="Arial" font-weight="700" text-anchor="middle" fill="#047857">Softmax</text><text x="445" y="64" font-size="9" font-family="Arial" text-anchor="middle" fill="#047857">Weights</text><path d="M520 60 H545" stroke="#64748B" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="545" y="30" width="150" height="60" rx="10" fill="#FCE7F3" stroke="#BE185D"/><text x="620" y="48" font-size="11" font-family="Arial" font-weight="700" text-anchor="middle" fill="#BE185D">Value</text><text x="620" y="64" font-size="9" font-family="Arial" text-anchor="middle" fill="#BE185D">Context</text></svg>


#### 1. Dot product Attention
The simplest scoring method computes the dot product between Query and Key.
$$Score(Q,K)=Q⋅K$$
> Same direction → large dot product

> Different direction → small dot product

> Complexity: $(n\times d_k)(d_k\times m)=O(n \times m)$

> Limitition: When $d_k$ becomes large (512, 1024, 4096), scores become extremely large. That leads to unstable Softmax.

#### 2. Scaled Dot Product Attention (Transformer)
**Scaled dot-product attention** measures similarity between a query and each key using a dot product. The scores are divided by $\sqrt{d_k}$ before applying softmax.

Without scaling, dot products can become large when the key dimension is high. Large values make softmax extremely concentrated, which can produce very small gradients. Scaling improves numerical stability and training.

$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

> Example:
If a query vector is similar to three key vectors with scores:

$[2.0, 1.0, 0.1]$

Softmax converts these scores into probabilities, assigning the largest weight to the first key. The output is a weighted average of the corresponding value vectors.

>Use-Case: This is the standard attention operation used in most modern Transformers, including:

- BERT
- GPT-style models
- T5
- Vision Transformers
- Multimodal Transformers

> Time complexity: For $L$ tokens and dimension $d$ -> $O(L^2d)$

The scaling factor changes the values of the scores but does not change the asymptotic complexity.


#### 3. Additive Attention (Bahdanau Attention)
**Additive attention**, also called **Bahdanau attention**, computes the compatibility between a query and key using a small feed-forward neural network rather than a dot product.

> A common form is: $e_{ij}=v^T\tanh(W_qq_i+W_kk_j+b)$

> Then: $\alpha_{ij}=\operatorname{softmax}_j(e_{ij})$

> The output is: $o_i=\sum_j\alpha_{ij}v_j$

- Additive attention works well even when queries and keys have different dimensions. It was especially influential in early neural machine-translation systems.
- While generating a translated word, the decoder combines its current hidden state with each encoder hidden state through a learned scoring network. It then focuses on the source words with the highest scores.

> Use cases

- Early neural machine translation
- Recurrent encoder–decoder networks
- Speech recognition
- Sequence-to-sequence prediction
- Situations where query and key dimensions differ

> Time complexity: For $n$ queries and $m$ keys, the number of query-key comparisons is $nm$. If the scoring network has internal dimension $a$, the scoring cost is approximately:

> $O(n\times m\times a)$

The value aggregation adds approximately $O(n\times m\times d_v)$. Thus, the total is often summarized as:

$O(n\times m(a+d_v))$

For equal sequence lengths, it is quadratic in sequence length: $O(L^2)$.


#### 4. Multiplicative Attention (Luong Attention)

**Multiplicative attention**, associated with Luong attention, computes similarity using matrix multiplication. Important variants include:

> Dot form : $e_{ij}=q_i^Tk_j$

> General form: $e_{ij}=q_i^TWk_j$

> Scaled dot form: $e_{ij}=\frac{q_i^Tk_j}{\sqrt{d_k}}$

The scaled version became the standard Transformer attention mechanism.

> Example: A decoder query representing the next target word is compared with every encoder key. The largest similarity scores identify the source words most relevant to generation.

> Use cases

- Neural machine translation
- Transformer models
- Retrieval systems
- Recommendation systems
- Matching queries against documents or features

> Time complexity: For \(n\) queries, \(m\) keys, and dimension \(d\):  $O(nmd)$

> For self-attention with \(n=m=L\): $O(L^2d)$

> Multiplicative attention is generally faster than additive attention because matrix multiplication is highly optimized on GPUs and accelerators.

## 2. Attention Relationship

### 1. Self-Attention
Self-attention allows every token to attend to every other token in the same sequence. That means that the queries, keys, and values all come from the same sequence.

For example, in the sentence:

> “The animal did not cross the street because it was tired.”

When processing “it,” self-attention may assign high weight to “animal,” helping the model resolve the reference.

> Input: $X = [x_1, x_2, x_3, x_4]$

> $Q = XW_Q$ , $K = XW_K$ , $V = XW_V$

> output:$\boxed{\operatorname{Attention}(Q, K, V) = \operatorname{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V}$

> “The cat sat on the mat.”

The representation of “sat” may attend strongly to “cat,” while “mat” may attend to “on” and “sat.” The model dynamically creates contextual representations for all tokens.

***`Self-attention is used in:`***
- Transformer language models such as BERT and GPT
- Machine translation
- Text classification
- Document summarization
- Vision Transformers, where image patches attend to one another
- Speech recognition

**`Time and space complexity:`**
For sequence length $L$ and hidden dimension $d$:

| Operation | Complexity |
|---|---:|
| Computing $QK^T$ | $O(L^2d)$ |
| Multiplying attention by $V$ | $O(L^2d)$ |
| Attention-score storage | $O(L^2)$ |
| Total attention computation | $O(L^2d)$ |

Self-attention provides excellent context modeling, but becomes expensive for long sequences.


### 2. Cross-Attention
In **cross-attention**, queries come from one sequence or modality, while keys and values come from another. It allows one representation to retrieve information from a different representation.

> For example, in an encoder–decoder translation model:
- The decoder produces the queries.
- The encoder output provides the keys and values.

The decoder can therefore focus on the relevant source-language words while generating each target-language word.

> Example- English-to-French translation:

> English input: “The house is large.”
> French output: “La maison est grande.”

When generating “maison,” the decoder may attend strongly to “house” in the encoder output.

> Use cases: Cross-attention is common in:
- Encoder–decoder Transformers such as T5 and the original Transformer
- Machine translation
- Text-to-image generation, where image features attend to text embeddings
- Image captioning
- Visual question answering
- Speech-to-text systems
- Retrieval-augmented generation

> Time complexity: If the decoder has $n$ query tokens and the encoder has $m$ key/value tokens: $O(n\times m\times d)$

> Space complexity for the attention matrix is: $O(n\times m)$

> Unlike self-attention, cross-attention is not necessarily quadratic in one sequence length, but it is still expensive when both sequences are long.

## 3. Information-access pattern
Information-access pattern is another independent way to classify attention.
> Which tokens are allowed to attend to which tokens?

> This is about the connections between tokens, not about how the score is calculated.


### 1. Full Attention
### 2. Causal/Mask Attention
**Causal attention** prevents a token from attending to future tokens. `It is used for autoregressive language generation.`. It is implemented with a triangular mask:

$M_{ij}=\begin{cases}
0 & j\le i\\
-\infty & j>i
\end{cases}$

After adding this mask to the attention scores, future positions receive probability zero after softmax.

![CA](../images/CA.svg)
- Only the lower triangle is visible.

> Example: When predicting the next word in:

> “The cat sat on the …”

The model may use “The,” “cat,” “sat,” and “on,” but it cannot use the future answer during training.

> Use cases
- Autoregressive language models
- Text generation
- Code completion
- Music generation
- Time-series forecasting
- Autoregressive image generation

> Time complexity: Although only about half of the $L^2$ score matrix is logically valid, the usual dense implementation still has: $O(L^2d)$ training-time complexity and $O(L^2)$ attention-score memory.

> During autoregressive inference, caching previous keys and values changes the cost per generated token. For a context of length $L$, one new token generally attends to $L$ previous tokens, costing approximately: $O(Ld)$ per generated token, excluding other model operations.

### 3. Bidirectional Attention
**Bidirectional attention** allows a token to attend to tokens on both its left and right sides. It is not restricted by a causal mask.

> Example:  "The bank approved the loan."

The meaning of “bank” can be inferred using words on both sides, especially “approved” and “loan.”

> Use cases
- BERT-style masked-language modeling
- Text classification
- Named-entity recognition
- Sentence similarity
- Extractive question answering
- Encoder components of sequence-to-sequence models

> Time complexity: Dense bidirectional attention has: $O(L^2d)$ computation and: $O(L^2)$ attention-score memory.

The main difference from causal attention is the connectivity pattern, not the asymptotic complexity.

### 4. Local Attention / Sliding window Attention
**Local attention** restricts each token to a neighborhood of nearby tokens instead of allowing it to attend to the entire sequence.

If each token attends to at most $w$ neighboring tokens, the attention matrix has approximately $Lw$ entries instead of $L^2$.

> Example: For a document or image, a token may attend only to the previous and next 128 tokens. Nearby context is often sufficient for local syntax, image edges, or short-range patterns.

> Use cases
- Long-document processing
- Speech recognition
- Long audio sequences
- Vision Transformers at high resolution
- Long video models
- BigBird-style and Longformer-style architectures

> Time complexity:For sequence length $L$, local window size $w$, and dimension $d$:

$O(Lwd)$

Space complexity is approximately:

$O(Lw)$

If $w$ is constant relative to $L$, the complexity is effectively linear in sequence length:

$O(Ld)$

The trade-off is that information may require multiple layers to travel across distant parts of the sequence.


### 5. Sparse Attention
**Sparse attention** uses a deliberately selected pattern of connections rather than a fully dense attention matrix. The pattern can include local windows, strided connections, global tokens, blocks, or random links.

> Example: A long-document model may use:

- Local attention for nearby words
- Global attention for special summary tokens
- Strided attention to connect distant sections

A document-summary token can attend broadly while ordinary tokens use local attention.

> Use cases
- Longformer
- BigBird
- Sparse image and video models
- Long-context language modeling
- Scientific and legal documents

> Time complexity:If each token attends to $s$ selected tokens:

$O(Lsd)$

with score memory:

$O(Ls)$

When $s\ll L$, this is much cheaper than dense attention. The actual complexity depends on the sparsity pattern and whether hardware can process it efficiently.

### 6. Global Attention

**Global attention** allows selected tokens to attend to all other tokens, while the rest may use local or sparse attention.

> Example: In a long document, a special token such as `[CLS]` may attend to every token to collect document-level information. Other tokens may attend only to nearby words.

> Use cases
- Document classification
- Longformer-style architectures
- Question-answering systems
- Image models with class tokens
- Hierarchical sequence models

> Time complexity:If there are $g$ global tokens and $L-g$ local tokens with window size $w$, a rough estimate is:

$O(Lwd+gLd)$

If $g$ and $w$ are small relative to $L$, this is close to linear:

$O(L(w+g)d)$

However, too many global tokens can make the method expensive.

## **4. Computational architecture**
> How can we reduce attention's computation and memory cost?

### 1. MHA (Multi-Head Attention)
**Multi-Head Attention** is an extension of the self-attention mechanism. **Multi-head attention** runs several attention operations in parallel. Each head learns a different representation subspace and may specialize in a different relationship.

![MHA](../images/mha.svg)

**How it works:**

1.  **Multiple "Heads":** The input $Q, K, V$ are split into $h$ different "heads" or subspaces. For each head, separate linear transformations are applied to the input embeddings to create distinct $Q_i, K_i, V_i$ matrices.
2.  **Parallel Attention:** The scaled dot-product attention function is then applied in parallel to each of these $h$ sets of $(Q_i, K_i, V_i)$. This results in $h$ different attention outputs.
3.  **Concatenation:** The outputs from all $h$ attention heads are concatenated together.
4.  **Linear Projection:** The concatenated output is then linearly projected once more (multiplied by a final weight matrix $W^O$) to produce the final output of the Multi-Head Attention layer. This projection brings the dimension back to the original input dimension.

> $MultiHead(Q, K, V) = Concat(head_1, ..., head_h)W^O$

> $head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)$

**Advantages of Multi-Head Attention:**

*   **Enriched Representation:** Allows the model to capture different types of relationships and dependencies. For example, one head might focus on syntactic dependencies, while another focuses on semantic relationships.
*   **Focus on Different Positions:** Each head can learn to attend to different parts of the input sequence, providing a more comprehensive understanding of the context.
*   **Stabilizes Training:** Averaging the results from multiple attention heads can lead to more stable and robust training.

**Example (Conceptual):**
In the sentence "The animal didn't cross the street because it was too tired."

*   One attention head might learn to associate "it" with "animal" (semantic link).
*   Another head might learn to associate "it" with "tired" (grammatical subject-verb agreement).
*   A third head might focus on the negation "didn't" and its scope.

By combining these different perspectives, Multi-Head Attention creates a richer and more nuanced contextual representation for each word.

> Time complexity: Suppose the total model dimension is $d$, divided among $h$ heads, so each head has dimension approximately $d/h$. The total attention computation is:

> $h\cdot O\left(L^2\frac{d}{h}\right)=O(L^2d)$

Thus, increasing the number of heads does not usually change the leading asymptotic complexity, although it affects constants, memory layout, and implementation efficiency.

> The projection layers add approximately: $O(Ld^2)$

> Therefore, a more complete Transformer-layer estimate is often: $O(L^2d+Ld^2)$

> Usage: Multi-head attention is central to:
- BERT and GPT
- Translation models
- Vision Transformers
- Audio and speech Transformers
- Multimodal architectures

### 2. MQA
Researchers discovered that most inference memory is consumed by Keys and Values, not Queries.

!['MQA'](../images/MQA.svg)

**Multi-query attention** uses many query heads but a single shared key head and value head.

This preserves some diversity in the queries while greatly reducing the amount of cached key/value data.

> Use cases

- Fast autoregressive decoding
- Large language models serving many users
- Memory-constrained inference
- Systems where decoding latency is important

> Time and space complexity: $O(L^2d)$ during full-sequence processing.

> The major benefit is memory and bandwidth reduction during generation. Key-value cache storage changes from approximately: $O(Lh_qd_h)$ to: $O(Ld_h)$

> for one shared key/value head, where $d_h$ is the head dimension.

### 3. GQA

!['MQA'](../images/GQA.svg)
### 4. MLA(Multi-head Latent Attention)
Multi-head Latent Attention (MLA) is a memory-efficient attention architecture introduced by DeepSeek that compresses the Key (K) and Value (V) representations into a low-dimensional latent space, dramatically reducing KV-cache memory while preserving the quality of standard Multi-Head Attention.

![MLA](../images/MLA.svg)

> 1. Input: $$x \in \mathbb{R}^{4096}$$

> 2. Compress: $z=Cx$
  - x = original hidden state (4096 dimensions)
  - C = learned compression matrix
  - z = latent vector (e.g., 512 dimensions)

> 3. Recover Key and Value: $K=Wk​_z$ ; $V=W_v​z$
  - These are lightweight projection layers.
  - K and V are reconstructed on demand—they are not permanently stored.

>4. Compute regular Attention.

### 5. Linear Attention


Standard attention computes:

$\operatorname{softmax}(QK^T)V$

The expensive part is forming the $L\times L$ matrix $QK^T$. **Linear attention** tries to rearrange or approximate the calculation so that keys and values are aggregated before interacting with each query.

> A simplified kernelized form is:
$\operatorname{Attention}(Q,K,V)_i
\approx
\frac{\phi(q_i)^T\left(\sum_j \phi(k_j)v_j^T\right)}
{\phi(q_i)^T\left(\sum_j \phi(k_j)\right)}$

Here, $\phi$ is a feature map that makes the attention operation associative.

![LA](../images/LA.svg)

> Example: Instead of comparing every token with every other token, the model summarizes all key-value interactions into an accumulated state. Each query then retrieves information from this state.

> Use cases
- Very long sequences
- Streaming speech recognition
- Online time-series prediction
- Long-context language modeling
- Memory-efficient sequence processing

> Time complexity: For sequence length $L$ and dimension $d$, a common estimate is: $O(Ld^2)$

> or, when the feature dimension is treated separately, $O(Ldr)$, where $r$ is the feature-map dimension.

> This is linear in $L$, but it may be less accurate than softmax attention because the attention kernel is approximated or replaced.

### Context Vector (Weighted Sum of Values)

Finally, the attention weights are multiplied by their corresponding **Value (V)** vectors. These weighted Value vectors are then summed up to produce the **context vector** for the current word. This context vector is a rich representation that incorporates information from all other words in the sequence, weighted by their relevance to the current word.

$ContextVector_i = \sum_{j=1}^{L} AttentionWeights_{ij} \cdot V_j$

Where $L$ is the length of the sequence.

This entire process is often summarized by the equation:

$Attention(Q, K, V) = Softmax(\frac{QK^T}{\sqrt{d_k}})V$

### Dry Run Example: Self-Attention for a single word

Let's consider a simplified example with a sentence: "I love NLP"

Assume we are calculating the self-attention output for the word "love".

1.  **Input Embeddings:** Each word is converted into an embedding vector.
    *   $E_I$, $E_{love}$, $E_{NLP}$

2.  **Generate Q, K, V Matrices:** For each word, we multiply its embedding by learned weight matrices $W^Q, W^K, W^V$ to get its Q, K, V vectors.
    *   $Q_{love} = E_{love} W^Q$
    *   $K_I = E_I W^K$, $K_{love} = E_{love} W^K$, $K_{NLP} = E_{NLP} W^K$
    *   $V_I = E_I W^V$, $V_{love} = E_{love} W^V$, $V_{NLP} = E_{NLP} W^V$

3.  **Calculate Dot Products (Scores) for $Q_{love}$:** We compare $Q_{love}$ with all Key vectors.
    *   $Score(love, I) = Q_{love} \cdot K_I$
    *   $Score(love, love) = Q_{love} \cdot K_{love}$
    *   $Score(love, NLP) = Q_{love} \cdot K_{NLP}$

4.  **Scale the Scores:** Divide each score by $\sqrt{d_k}$.
    *   $ScaledScore(love, I) = \frac{Q_{love} \cdot K_I}{\sqrt{d_k}}$
    *   $ScaledScore(love, love) = \frac{Q_{love} \cdot K_{love}}{\sqrt{d_k}}$
    *   $ScaledScore(love, NLP) = \frac{Q_{love} \cdot K_{NLP}}{\sqrt{d_k}}$

5.  **Apply Softmax:** Convert scaled scores into attention weights.
    *   $Weight(love, I) = Softmax(ScaledScore(love, I))$
    *   $Weight(love, love) = Softmax(ScaledScore(love, love))$
    *   $Weight(love, NLP) = Softmax(ScaledScore(love, NLP))$
    (These weights sum to 1)

6.  **Compute Context Vector for "love":** Multiply each Value vector by its corresponding attention weight and sum them up.
    *   $ContextVector_{love} = Weight(love, I) \cdot V_I + Weight(love, love) \cdot V_{love} + Weight(love, NLP) \cdot V_{NLP}$

This $ContextVector_{love}$ is the output of the self-attention layer for the word "love", enriched with contextual information from "I" and "NLP" based on their relevance. This process is performed in parallel for every word in the sequence.

##


### Feed Forward Network
After the Multi-Head Attention layer, each position in the Transformer's encoder and decoder blocks passes through a simple, position-wise **Feed-Forward Network (FFN)**. This FFN is applied independently and identically to each position. It consists of two linear transformations with a ReLU activation in between.

$FFN(x) = max(0, xW_1 + b_1)W_2 + b_2$

Where:
*   $x$ is the output from the Multi-Head Attention layer for a specific position.
*   $W_1, b_1, W_2, b_2$ are learned parameters. Importantly, these parameters are the same for every position, but they are different for each layer of the Transformer.

**Purpose of the Feed-Forward Network:**

*   **Non-linearity:** The FFN introduces non-linearity into the model, allowing it to learn more complex patterns than a purely linear model. The ReLU activation function is key here.
*   **Feature Transformation:** It transforms the representation of each position independently. While self-attention allows information to flow between different positions, the FFN processes each position's representation in isolation, allowing it to learn complex, position-specific feature transformations.
*   **Increased Model Capacity:** The FFN contributes significantly to the model's capacity to learn and represent intricate relationships within the data.

**Example:**
Imagine the output from the Multi-Head Attention layer for the word "bank" (which has been contextualized by other words in the sentence). This contextualized vector for "bank" is then fed into the FFN. The FFN might learn to emphasize certain features within this vector that are relevant for disambiguating its meaning (e.g., whether it refers to a financial institution or a river bank), or for preparing it for subsequent layers.

### Residual Connection (Skip Connection)

Both the Multi-Head Attention sub-layer and the Feed-Forward Network sub-layer in the Transformer architecture are followed by a **Residual Connection**, also known as a **Skip Connection**. This concept, popularized by ResNet architectures in computer vision, involves adding the input of a sub-layer directly to its output.

Mathematically, if $X$ is the input to a sub-layer and $Sublayer(X)$ is the function computed by the sub-layer, the output with a residual connection is:

$Output = X + Sublayer(X)$

**Purpose of Residual Connections:**

*   **Mitigate Vanishing Gradients:** In deep neural networks, gradients can vanish as they propagate backward through many layers. Residual connections provide a direct path for gradients to flow, allowing them to bypass non-linear transformations and reach earlier layers more effectively. This helps in training very deep networks.
*   **Facilitate Identity Mapping:** They allow the network to easily learn an identity function. If a sub-layer doesn't need to transform its input, it can simply learn to output zero, and the residual connection ensures the input is passed through unchanged. This makes it easier for the network to learn incremental changes rather than entirely new transformations.
*   **Improved Training Stability:** By providing alternative paths for information flow, residual connections contribute to more stable training and faster convergence of deep models.

**Example:**
Consider a deep Transformer model with many layers. Without residual connections, if a particular attention or FFN sub-layer is not learning effectively, the information might degrade significantly as it passes through. With a residual connection, the original input signal is preserved and added to the sub-layer's output, ensuring that at least the original information is available to the next layer, even if the sub-layer's transformation is not perfect.

### Layer Normalization

Following each sub-layer (Multi-Head Attention and Feed-Forward Network) in the Transformer, and *after* the residual connection, **Layer Normalization** is applied. This technique normalizes the inputs across the features for each sample independently, rather than across a batch of samples (as in Batch Normalization).

Mathematically, for an input $x$ to a layer, Layer Normalization computes the mean ($\mu$) and variance ($\sigma^2$) of the features for each individual sample:

$\mu = \frac{1}{D} \sum_{i=1}^{D} x_i$
$\sigma^2 = \frac{1}{D} \sum_{i=1}^{D} (x_i - \mu)^2$

Then, the normalized output $y$ is calculated as:

$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$

Where:
*   $D$ is the number of features (e.g., $d_{model}$).
*   $\epsilon$ is a small constant to prevent division by zero.
*   $\gamma$ and $\beta$ are learned scaling and shifting parameters, respectively, allowing the network to restore the original representation if needed.

**Purpose of Layer Normalization:**

*   **Stabilizes Training:** Normalizing the activations helps to stabilize the training process, especially in deep networks, by keeping the input distribution to subsequent layers consistent. This prevents issues like exploding or vanishing activations.
*   **Faster Convergence:** By reducing internal covariate shift (the change in the distribution of network activations due to the change in network parameters during training), Layer Normalization allows for higher learning rates and faster convergence.
*   **Independent of Batch Size:** Unlike Batch Normalization, Layer Normalization computes statistics independently for each training example. This makes it particularly well-suited for RNNs and Transformers, where batch sizes can vary or be small, and for tasks where sequence lengths differ within a batch.

**Example:**
Consider a vector representing a word after it has passed through a Multi-Head Attention layer and had its residual connection added. This vector has $d_{model}$ dimensions. Layer Normalization will calculate the mean and variance across these $d_{model}$ dimensions for *that single word vector*. It then normalizes these values and applies learned scaling and shifting, ensuring that the features for that word are in a stable range before being passed to the next sub-layer or block. This process is applied independently to every word vector in the sequence.

## Encoder Block

The Transformer architecture is composed of a stack of identical **Encoder Blocks**. Each encoder block is responsible for processing a portion of the input sequence and transforming it into a richer, contextualized representation. The original Transformer model uses 6 such encoder blocks stacked on top of each other.

!["ER"](../images/E-R.svg)

Each Encoder Block consists of two main sub-layers:

1.  **Multi-Head Self-Attention Mechanism:** This sub-layer allows the encoder to weigh the importance of different words in the input sequence when processing each word. It computes a contextualized representation for each input token by attending to all other tokens in the sequence. As described previously, this involves the Query, Key, and Value transformations, dot-product attention, scaling, softmax, and concatenation of multiple attention heads.

2.  **Position-wise Feed-Forward Network (FFN):** This sub-layer is applied independently to each position in the sequence. It consists of two linear transformations with a ReLU activation in between, providing non-linearity and allowing the model to learn complex, position-specific feature transformations.

Crucially, each of these two sub-layers is followed by a **Residual Connection** and then **Layer Normalization**.

$Output_{EncoderBlock} = LayerNorm(Input + Sublayer(Input))$

**Flow within an Encoder Block:**

1.  **Input:** The input to the first encoder block is the sum of the word embeddings and their corresponding positional encodings. For subsequent encoder blocks, the input is the output of the previous encoder block.
2.  **Multi-Head Self-Attention:** The input passes through the Multi-Head Self-Attention layer. This layer computes a new representation for each token, incorporating information from all other tokens in the sequence.
3.  **Add & Norm 1:** The output of the Multi-Head Self-Attention layer is added to its input (residual connection), and then Layer Normalization is applied.
4.  **Feed-Forward Network:** The normalized output then passes through the Position-wise Feed-Forward Network.
5.  **Add & Norm 2:** The output of the FFN is added to its input (residual connection), and then Layer Normalization is applied.
6.  **Output:** The final output of the encoder block is a sequence of contextualized representations, one for each input token. This output then becomes the input for the next encoder block in the stack, or the input to the decoder.

**Example (Conceptual):**
Imagine an encoder block processing the sentence "The cat sat on the mat." For the word "cat", the Multi-Head Self-Attention layer will consider "The", "sat", "on", "the", "mat" to form a richer representation of "cat" that understands its role as the subject. This contextualized "cat" representation then goes through the FFN to further refine its features. This process happens for all words, and the output is a new sequence of vectors, where each vector is a highly contextualized representation of its corresponding input word.

### Decoder Block

The Transformer architecture also consists of a stack of identical **Decoder Blocks**, typically 6 of them, similar to the encoder. The decoder's role is to generate the output sequence one token at a time, taking the encoder's output (the contextualized representations of the input sequence) and the previously generated output tokens as input.

![DR](../images/D-R.svg)

Each Decoder Block is more complex than an Encoder Block, consisting of three main sub-layers:

1.  **Masked Multi-Head Self-Attention:** This sub-layer is similar to the encoder's self-attention but with a crucial modification: it is **masked**. This masking ensures that when predicting the next token, the decoder can only attend to previously generated tokens and the current token, not future tokens. This prevents information leakage from future tokens during training, maintaining the auto-regressive property of sequence generation.

2.  **Multi-Head Cross-Attention (Encoder-Decoder Attention):** This sub-layer allows the decoder to attend to the output of the encoder stack. Here, the Query (Q) comes from the *decoder's* previous layer, while the Key (K) and Value (V) come from the *encoder's* output. This mechanism enables the decoder to focus on relevant parts of the input sequence when generating each output token, similar to how attention worked in the original Seq2Seq models.

3.  **Position-wise Feed-Forward Network (FFN):** Similar to the encoder, this sub-layer is applied independently to each position, providing non-linearity and feature transformation.

Again, each of these three sub-layers is followed by a **Residual Connection** and then **Layer Normalization**.

**Flow within a Decoder Block:**

1.  **Input:** The input to the first decoder block is the sum of the previously generated output token embeddings and their corresponding positional encodings. For subsequent decoder blocks, the input is the output of the previous decoder block.
2.  **Masked Multi-Head Self-Attention:** The input passes through this layer. Each token attends to all preceding tokens in the *decoder's own output sequence*.
3.  **Add & Norm 1:** Residual connection and Layer Normalization are applied.
4.  **Multi-Head Cross-Attention:** The output from the previous step (decoder's contextualized representation) acts as the Query. The Key and Value come from the final output of the encoder stack. This allows the decoder to attend to the source input.
5.  **Add & Norm 2:** Residual connection and Layer Normalization are applied.
6.  **Feed-Forward Network:** The output then passes through the Position-wise Feed-Forward Network.
7.  **Add & Norm 3:** Residual connection and Layer Normalization are applied.
8.  **Output:** The final output of the decoder block is a sequence of contextualized representations, which are then typically passed to a final linear layer and softmax function to predict the probability distribution over the vocabulary for the next token.

**Example (Conceptual):**
Imagine the decoder generating a translation for "The cat sat on the mat." When it's about to generate "chat" (cat in French):

*   The **Masked Self-Attention** would ensure it only looks at "Le" (The) and the start-of-sequence token, but not at "était" (was) or "assis" (sat) yet.
*   The **Cross-Attention** would then take the decoder's current state (contextualized by "Le") and query the encoder's output. It would likely assign high attention weights to the encoder's representation of "The" and "cat" from the English input, helping it decide to generate "chat".

This iterative process, guided by both its own generated history and the full source input, allows the decoder to generate coherent and contextually relevant output sequences.

### Complete Forward Pass

The **Complete Forward Pass** of a Transformer model involves the sequential execution of its Encoder and Decoder components to transform an input sequence into an output sequence. This process is fundamental to how Transformers perform tasks like machine translation, text summarization, and question answering.

#### Encoder Forward Pass:

1.  **Input Preparation:** The input sequence (e.g., source sentence) is first converted into **word embeddings**. These embeddings are then combined with **positional encodings** to inject information about the word order. This combined representation forms the initial input to the encoder stack.

2.  **Encoder Stack Processing:** The prepared input sequence passes through a stack of $N$ identical **Encoder Blocks** (typically $N=6$).
    *   Each Encoder Block processes the sequence through its **Multi-Head Self-Attention** layer, followed by a **Feed-Forward Network (FFN)**.
    *   Crucially, each sub-layer (Self-Attention and FFN) is wrapped with a **Residual Connection** and **Layer Normalization**.
    *   The Multi-Head Self-Attention allows each word to attend to all other words in the input sequence, creating a rich contextual representation.
    *   The FFN further processes these contextualized representations independently for each position.

3.  **Encoder Output:** The final output of the encoder stack is a sequence of contextualized representations (often called memory keys and values) for each word in the input sequence. This output is then passed to the decoder.

#### Decoder Forward Pass:

1.  **Input Preparation:** The decoder receives two main inputs:
    *   The **encoder's output** (memory keys and values).
    *   The **previously generated output sequence** (e.g., target sentence generated so far), which is also converted into word embeddings and combined with positional encodings. During training, this is the actual target sequence shifted right (so the model predicts the next token). During inference, it starts with a special `[START]` token and iteratively appends its own predictions.

2.  **Decoder Stack Processing:** The prepared decoder input passes through a stack of $N$ identical **Decoder Blocks** (typically $N=6$).
    *   Each Decoder Block contains three sub-layers:
        *   **Masked Multi-Head Self-Attention:** This layer allows the decoder to attend to previously generated tokens in its *own output sequence*. The masking ensures that attention is only paid to past tokens, preserving the auto-regressive property.
        *   **Multi-Head Cross-Attention (Encoder-Decoder Attention):** This layer takes Queries from the decoder's masked self-attention output and Keys/Values from the encoder's final output. This allows the decoder to focus on relevant parts of the *input sequence* when generating the next output token.
        *   **Feed-Forward Network (FFN):** Similar to the encoder, this FFN processes the contextualized representations independently for each position.
    *   Again, each sub-layer is followed by a **Residual Connection** and **Layer Normalization**.

3.  **Output Layer:** The final output of the decoder stack is passed through a linear layer, followed by a softmax function. This produces a probability distribution over the entire vocabulary for the next token in the output sequence.

4.  **Token Prediction:** During training, the predicted probabilities are compared to the actual next token to calculate the loss. During inference, the token with the highest probability is selected as the next word, and this word is then fed back into the decoder as part of the input for the next time step, until an `[END]` token is generated or a maximum length is reached.

#### Dry Run Example: Translating "Hello world" to French "Bonjour le monde"

**Assumptions:**
*   Input sequence: `[START] Hello world [END]`
*   Target sequence (for training): `[START] Bonjour le monde [END]`
*   Vocabulary includes `Hello`, `world`, `Bonjour`, `le`, `monde`, `[START]`, `[END]`
*   $N=1$ Encoder Block, $N=1$ Decoder Block for simplicity.

**Encoder Forward Pass:**

1.  **Input:** `[START] Hello world [END]` is converted to embeddings + positional encodings.
    *   `E_start + PE_0`, `E_hello + PE_1`, `E_world + PE_2`, `E_end + PE_3`
2.  **Encoder Block:** These combined vectors pass through the Encoder Block.
    *   **Multi-Head Self-Attention:** Each token attends to all other tokens. `E_hello` now contains information about `world` and `[START]`, etc.
    *   **FFN:** Further transforms these contextualized representations.
    *   **Output:** `C_start`, `C_hello`, `C_world`, `C_end` (contextualized representations of the input sequence).

**Decoder Forward Pass (Inference - generating output one token at a time):**

*   **Step 1: Predict "Bonjour"**
    1.  **Decoder Input:** `[START]` token embedding + positional encoding (`E_start_dec + PE_0`).
    2.  **Decoder Block:**
        *   **Masked Self-Attention:** `E_start_dec + PE_0` attends only to itself (as it's the only token so far).
        *   **Cross-Attention:** Queries from the masked self-attention output attend to `C_start`, `C_hello`, `C_world`, `C_end` from the encoder. It learns to focus on `C_hello` and `C_world` to predict the first word.
        *   **FFN:** Processes the output.
    3.  **Output Layer:** Linear layer + Softmax predicts `Bonjour` with highest probability.

*   **Step 2: Predict "le"**
    1.  **Decoder Input:** `[START] Bonjour` token embeddings + positional encodings (`E_start_dec + PE_0`, `E_bonjour + PE_1`).
    2.  **Decoder Block:**
        *   **Masked Self-Attention:** `E_bonjour + PE_1` attends to `E_start_dec + PE_0` and itself.
        *   **Cross-Attention:** Queries from the masked self-attention output attend to `C_start`, `C_hello`, `C_world`, `C_end`. It learns to focus on `C_hello` and `C_world` again, but now in the context of `Bonjour` already being generated.
        *   **FFN:** Processes the output.
    3.  **Output Layer:** Linear layer + Softmax predicts `le` with highest probability.

*   **Step 3: Predict "monde"**
    1.  **Decoder Input:** `[START] Bonjour le` token embeddings + positional encodings (`E_start_dec + PE_0`, `E_bonjour + PE_1`, `E_le + PE_2`).
    2.  **Decoder Block:**
        *   **Masked Self-Attention:** `E_le + PE_2` attends to `E_start_dec + PE_0`, `E_bonjour + PE_1` and itself.
        *   **Cross-Attention:** Queries attend to encoder outputs. Focus shifts to `C_world` to predict `monde`.
        *   **FFN:** Processes the output.
    3.  **Output Layer:** Linear layer + Softmax predicts `monde` with highest probability.

*   **Step 4: Predict "[END]"**
    1.  **Decoder Input:** `[START] Bonjour le monde` token embeddings + positional encodings.
    2.  **Decoder Block:** Similar process.
    3.  **Output Layer:** Linear layer + Softmax predicts `[END]` with highest probability.

**Final Output:** `Bonjour le monde`

This iterative process, where the decoder generates one token at a time and feeds its own output back as input, is how the Transformer generates sequences. The parallel nature of self-attention within each block, however, allows for much faster computation compared to traditional RNNs during both training and inference (when generating multiple tokens in parallel within a batch).

## Interview Questions and Answers


### Self-Attention Core

**Q1: What is the fundamental idea behind Self-Attention, and how does it differ from traditional attention mechanisms in RNN-based Seq2Seq models?**

**A1:** Self-Attention allows each element in a sequence to attend to all other elements within the *same* sequence, computing a contextualized representation for each. This differs from traditional attention, where a decoder attends to an encoder (cross-attention), focusing on relevant parts of a *different* sequence. Self-attention enables direct capture of long-range dependencies within a single sequence, regardless of token distance.

**Tricky Follow-up:** If self-attention allows every word to attend to every other word, what is the computational complexity for a sequence of length L, and what are the practical implications for very long sequences?

**Q2: Explain the roles of Query (Q), Key (K), and Value (V) in the Self-Attention mechanism.**

**A2:**
*   **Query (Q):** Represents what we are looking for. For a given word, its Q vector is used to query all other words.
*   **Key (K):** Represents what each word contains. Each word's K vector is compared against the Q of other words to determine relevance.
*   **Value (V):** Represents the actual content or information of each word. Once relevance is determined, the V vectors are weighted and summed to form the output.

**Tricky Follow-up:** Why are Q, K, and V derived from the same input embedding but using different weight matrices? What would happen if they all used the same weight matrix?

**Q3: Why is scaling by $\sqrt{d_k}$ important in the scaled dot-product attention?**

**A3:** Scaling the dot products by $\sqrt{d_k}$ (the square root of the dimension of the Key vectors) is crucial for stabilizing the training process. Without scaling, the dot products can become very large, especially with large $d_k$. This can push the softmax function into regions where its gradients are extremely small, leading to vanishing gradients and hindering effective learning.

**Tricky Follow-up:** What would be the consequence if you omitted the scaling factor, and how might this manifest during model training?

### Structural Components

**Q4: What is Multi-Head Attention, and what advantages does it offer over single-head attention?**

**A4:** Multi-Head Attention is an extension where the input Q, K, V are linearly projected multiple times ($h$ heads) into different representation subspaces. Attention is then performed in parallel for each head, and their outputs are concatenated and linearly projected. Advantages include: (1) **Enriched Representation:** Captures different types of relationships (e.g., syntactic vs. semantic). (2) **Focus on Different Positions:** Each head can learn to attend to different parts of the input. (3) **Stabilizes Training:** Averaging across heads leads to more robust learning.

**Tricky Follow-up:** How does the total number of parameters in Multi-Head Attention compare to a single-head attention mechanism with the same output dimension? Is it more or less efficient in terms of parameter count?

**Q5: Why are Positional Encodings necessary in the Transformer architecture, and how are they typically generated?**

**A5:** Positional Encodings are necessary because the Transformer lacks recurrence or convolutions, meaning it inherently loses information about the order of tokens in a sequence. Word order is crucial for language understanding. They are typically generated using fixed sine and cosine functions of different frequencies, which are then added to the input word embeddings. This provides unique positional information for each token and allows the model to generalize to longer sequences.

**Tricky Follow-up:** If you were to replace sinusoidal positional encodings with learned positional embeddings, what are the potential benefits and drawbacks, especially concerning generalization to unseen sequence lengths?

**Q6: Describe the purpose of the Feed-Forward Network (FFN) within a Transformer block.**

**A6:** The FFN is a simple, position-wise network applied independently to each position after the attention sub-layer. It consists of two linear transformations with a ReLU activation in between. Its purpose is to introduce **non-linearity** into the model, allowing it to learn more complex patterns, and to perform **feature transformation** on the contextualized representation of each token in isolation.

**Tricky Follow-up:** Why is the FFN applied position-wise and independently to each token, rather than across the entire sequence like attention? What does this imply about the FFN's role compared to attention?

### Training & Stability

**Q7: What is a Residual Connection, and why is it important for training deep Transformer models?**

**A7:** A Residual Connection (or skip connection) adds the input of a sub-layer directly to its output ($Output = Input + Sublayer(Input)$). It is crucial for training deep Transformer models because it: (1) **Mitigates Vanishing Gradients:** Provides a direct path for gradients to flow, preventing them from diminishing in deep networks. (2) **Facilitates Identity Mapping:** Allows sub-layers to easily learn an identity function, making it easier for the network to learn incremental changes. (3) **Improves Training Stability** and faster convergence.

**Tricky Follow-up:** How does the concept of residual connections relate to the idea of
Tricky Follow-up: How does the concept of residual connections relate to the idea of enabling deeper networks, and what problem does it specifically solve that makes deeper networks feasible?

**Q8: Explain the role of Layer Normalization in the Transformer architecture, and how it differs from Batch Normalization.**

**A8:** Layer Normalization normalizes the inputs across the features for each individual sample independently. It computes the mean and variance of the features for each token vector and then normalizes them, applying learned scaling and shifting parameters. Its role is to **stabilize training** by keeping the input distribution to subsequent layers consistent, preventing exploding/vanishing activations, and leading to **faster convergence**. It differs from Batch Normalization, which normalizes across the batch dimension for each feature, making Layer Normalization more suitable for variable-length sequences and smaller batch sizes common in NLP.

**Tricky Follow-up:** In what specific scenarios would Layer Normalization be preferred over Batch Normalization in a sequence model, and why is this distinction particularly important for Transformers?

### Architecture: Encoder and Decoder Blocks

**Q9: Describe the main components of an Encoder Block in the Transformer.**

**A9:** An Encoder Block consists of two main sub-layers: a **Multi-Head Self-Attention mechanism** and a **Position-wise Feed-Forward Network (FFN)**. Each of these sub-layers is followed by a **Residual Connection** and **Layer Normalization**. The Multi-Head Self-Attention allows the encoder to create contextualized representations by attending to all words in the input sequence, while the FFN applies non-linear transformations independently to each position.

**Tricky Follow-up:** If you were to remove the FFN from the Encoder Block, what capabilities would the Transformer lose, and how would this impact its ability to learn complex patterns?

**Q10: What are the three main sub-layers in a Decoder Block, and what is the purpose of Masked Multi-Head Self-Attention?**

**A10:** A Decoder Block has three main sub-layers: **Masked Multi-Head Self-Attention**, **Multi-Head Cross-Attention (Encoder-Decoder Attention)**, and a **Position-wise Feed-Forward Network (FFN)**. Each is followed by a Residual Connection and Layer Normalization. The **Masked Multi-Head Self-Attention** is crucial because it ensures that when the decoder is predicting the next token, it can only attend to previously generated tokens and the current token, but *not* to future tokens in the output sequence. This maintains the auto-regressive property required for sequence generation.

**Tricky Follow-up:** Why is the cross-attention layer in the decoder *not* masked, unlike the self-attention layer? What would be the consequence if it were masked?




## Architecture
```text
Transformer Block
        │
        ▼
Multi-Head Attention
        │
        ├── Head 1
        │      ├── Q
        │      ├── K
        │      ├── V
        │      └── Attention(Q,K,V)
        │
        ├── Head 2
        │      ├── Q
        │      ├── K
        │      ├── V
        │      └── Attention(Q,K,V)
        │
        ├── ...
        │
        └── Head h
               ├── Q
               ├── K
               ├── V
               └── Attention(Q,K,V)
        │
        ▼
Concatenate Heads
        │
        ▼
Output Projection
        │
        ▼
Feed Forward Network
```


### Q1. What is an attention head?

**Answer:**  
An attention head is one independent attention mechanism with its own learnable projection matrices (WQ, WK, WV). Each head computes its own attention distribution and context vectors, allowing the model to learn different types of relationships in parallel.

---

### Q2. Why do we need multiple heads instead of one large head?

**Answer:**  
Multiple heads let the model learn different kinds of patterns simultaneously. For example, one head may focus on syntactic relationships, another on semantic similarity, and another on long-range dependencies. Their outputs are concatenated and projected back into the model dimension.

---

### Q3. What is the difference between attention scores and attention weights?

**Answer:**  
- **Attention scores** are the raw similarities computed as **QKᵀ / √dₖ**.  
- **Attention weights** are the probabilities obtained by applying **Softmax** to those scores. These weights determine how much each Value vector contributes to the output.

---

### Q4. What is the difference between a head and Multi-Head Attention?

**Answer:**  
A **head** is a single attention computation with its own Q/K/V projections. **Multi-Head Attention** is a collection of multiple heads running in parallel, whose outputs are concatenated and passed through an output projection.

---

### Q5. What is the difference between self-attention and cross-attention?

**Answer:**  
- In **self-attention**, Q, K, and V are all computed from the same input sequence.  
- In **cross-attention**, Q comes from one sequence (typically the decoder), while K and V come from another sequence (typically the encoder output).